#### Pydantic Basics: Creating and Using Models
Pydantic models are the foundation of data validation in Python. They use Python type annotations to define the structure and validate data at runtime. Here's a detailed exploration of basic model creation with several examples.



In [1]:
from pydantic import BaseModel

In [2]:
from dataclasses import dataclass

@dataclass
class Person():
    name:str
    age:int
    city:str

person=Person(name="Suraj",age=35,city="Pune")
print(person)


Person(name='Suraj', age=35, city='Pune')


In [3]:
person=Person(name="Suraj",age=35,city=35)
print(person)

Person(name='Suraj', age=35, city=35)


In [4]:
class Person1(BaseModel):
    name:str
    age:int
    city:str

person=Person1(name="Suraj",age=35,city="Pune")
print(person)




name='Suraj' age=35 city='Pune'


In [5]:
person1=Person1(name="Suraj",age=35,city=12)
print(person1)

ValidationError: 1 validation error for Person1
city
  Input should be a valid string [type=string_type, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

#### 2. Model with Optional Fields
Add optional fields using Python's Optional type:



In [6]:
from typing import Optional
class Employee(BaseModel):
    id: int
    name: str
    department: str
    salary: Optional[float] = None  # Optional with default value
    is_active: Optional[bool] = True  # Optional with default True


In [7]:
# Examples with and without optional fields
emp1 = Employee(id=1, name="John", department="IT")
print(emp1)  # id=1 name='John' department='IT' salary=None is_active=True

id=1 name='John' department='IT' salary=None is_active=True


In [8]:
emp2 = Employee(id=2, name="Jane", department="HR", salary=60000, is_active=False)
print(emp2)

id=2 name='Jane' department='HR' salary=60000.0 is_active=False


Definition:
- Optional[type]: Indicates the field can be None

- Default value (= None or = True): Makes the field optional

- Required fields must still be provided

- Pydantic validates types even for optional fields when values are provided



In [9]:
from pydantic import BaseModel
from typing import List

class Classroom(BaseModel):
    room_number: str
    students: List[str]  # List of strings
    capacity: int

In [10]:
# Create a classroom
classroom = Classroom(
    room_number="A101",
    students=("Alice", "Bob", "Charlie"),
    capacity=30
)
print(classroom)

room_number='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30


In [11]:
try:
    invalid_val=Classroom(room_number="A1",students=["Suraj",123],capacity=30)
except ValueError as e:
    print(e)

1 validation error for Classroom
students.1
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


#### 4. Model with Nested Models
Create complex structures with nested models:



In [12]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    zip_code: int

class Customer(BaseModel):
    customer_id: int
    name: str
    address: Address  # Nested model

# Create a customer with nested address
customer = Customer(
    customer_id=1,
    name="Emma",
    address={"street": "123 Main St", "city": "Boston", "zip_code": "02108"}
)
print(customer)

customer_id=1 name='Emma' address=Address(street='123 Main St', city='Boston', zip_code=2108)


#### Pydantic Fields: Customization and Constraints

The Field function in Pydantic enhances model fields beyond basic type hints by allowing you to specify validation rules, default values, aliases, and more. Here's a comprehensive tutorial with examples.





In [15]:
from pydantic import BaseModel,Field
class Item(BaseModel):
    name:str=Field(min_length=2,max_length=50)
    price:float= Field(gt=0,le=1000) #greater than 0, less than or equal to 1000
    quantity:int=Field(ge=0)

# Valid instance
item = Item(name="Book", price=10, quantity=10)

print(item)


name='Book' price=10.0 quantity=10


In [16]:
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(..., description="Unique username for the user")
    age: int = Field(default=18, description="User age, defaults to 18")
    email: str = Field(default_factory=lambda: "user@example.com", description="Default email address")

# Examples
user1 = User(username="alice")
print(user1)  # username='alice' age=18 email='user@example.com'

user2 = User(username="bob", age=25, email="bob@domain.com")
print(user2)  # username='bob' age=25 email='bob@domain.com'

username='alice' age=18 email='user@example.com'
username='bob' age=25 email='bob@domain.com'


In [ ]:
print(User.model_json_schema())

{'properties': {'username': {'description': 'Unique username for the user', 'title': 'Username', 'type': 'string'}, 'age': {'default': 18, 'description': 'User age, defaults to 18', 'title': 'Age', 'type': 'integer'}, 'email': {'description': 'Default email address', 'title': 'Email', 'type': 'string'}}, 'required': ['username'], 'title': 'User', 'type': 'object'}


## Advanced Pydantic Concepts

This section expands on key Pydantic features (primarily Pydantic v2) and practical patterns you'll use in production-grade code.

- BaseModel: core class for schema definition and validation. Use type annotations to declare fields.
- Field: add validation rules (gt, ge, lt, le, min_length, max_length), descriptions, and metadata.
- Parsing & Validation APIs:
  - model_validate / model_validate_json — create a model from dict/json with validation (v2).
  - model_dump / model_dump_json — serialize model to dict/json (v2).
- Validators:
  - @field_validator — validate or coerce individual fields (supports mode='before'/'after').
  - @model_validator — validate or transform the whole model (v2 replacement for root_validator).
- Constrained types / Annotated: enforce fine-grained constraints (min/max, regex) via Field or Annotated.
- Nested models: compose complex schemas by embedding models as field types.
- Error handling: Pydantic raises `ValidationError` with detailed messages; catch and inspect `.errors()` for programmatic handling.
- Version notes: Pydantic v2 introduced API changes (model_validate, model_dump, field_validator, model_validator). If you still use v1, the equivalent methods are `parse_obj`, `dict()`, `validator`, and `root_validator`.

Tips:
- Prefer explicit types (List[str], dict[str,int]) over bare `list`/`dict` for clearer validation and schema generation.
- Avoid heavy logic in validators — keep them deterministic and fast.



In [ ]:
# Practical Pydantic Examples (v2-style APIs)
from pydantic import BaseModel, Field, field_validator

class UserModel(BaseModel):
    username: str = Field(..., min_length=2, description="User's display name")
    age: int = Field(ge=0, description="Age in years")
    email: str | None = None

    @field_validator('username', mode='before')
    def strip_username(cls, v):
        # coerce and clean the username before validation
        if isinstance(v, str):
            return v.strip()
        return v

    @field_validator('age', mode='before')
    def coerce_age(cls, v):
        # allow numeric strings, will be converted to int
        return int(v)

# Validate from a dict (model_validate is v2 equivalent of parse_obj)
raw = {"username": "  alice  ", "age": "25"}
user = UserModel.model_validate(raw)
print(user)
print("Dump as dict:", user.model_dump())
print("Dump as JSON:", user.model_dump_json())

# Nested / complex example
class Address(BaseModel):
    street: str
    city: str
    zip_code: int

class Customer(BaseModel):
    id: int
    name: str
    address: Address

c = Customer.model_validate({
    "id": 1,
    "name": "Bob",
    "address": {"street": "1 Road", "city": "Pune", "zip_code": 411001}
})
print(c)

# Handling validation errors
from pydantic import ValidationError
try:
    bad = UserModel.model_validate({"username": "x", "age": -5})
except ValidationError as e:
    print("Validation failed:")
    print(e)

# NOTE: If you're using Pydantic v1, use parse_obj(), dict(), and @validator instead.


### Best Practices & Tips

- Use explicit types: prefer `List[str]`, `dict[str,int]` over `list`/`dict` for predictable validation.
- Keep validators simple and fast; avoid network calls or heavy computation in validators.
- Validate inputs at boundary points (API layer) and use Pydantic for configuration parsing and request validation.
- Use `.model_validate()` (v2) for untrusted input and `.model_dump()` to serialize output.
- For environment/config settings, prefer dedicated settings packages (`pydantic-settings` for v2) to load from env vars with strong typing.
- Use `ValidationError.errors()` to programmatically extract location and type of errors for API error responses.
- Write unit tests for models: ensure both valid and invalid inputs behave as expected.
- When supporting both Pydantic v1 & v2 in documentation or code, clearly annotate which API is used.



## Interview Questions & Answers — Pydantic (Quick Reference)

1. Q: What is Pydantic and why use it?
   A: Pydantic is a data validation and settings management library that uses Python type hints to validate data at runtime. It reduces boilerplate, provides clear error messages, and integrates tightly with tools like FastAPI.

2. Q: How do you define an optional field in Pydantic?
   A: Use typing.Optional (e.g., `field: Optional[str] = None`) or a default value. Pydantic still validates types when a value is provided.

3. Q: How do you run custom validation on a field?
   A: In v2 use `@field_validator('field_name')`; in v1 use `@validator('field_name')`.

4. Q: How do you validate complex nested structures?
   A: Compose models by nesting BaseModel classes as field types — Pydantic will validate nested models recursively.

5. Q: How to serialize a Pydantic model to JSON?
   A: In v2 use `model.model_dump_json()`; in v1 use `model.json()` or `model.dict()`.

6. Q: How do you get machine-friendly error details for API responses?
   A: Catch `ValidationError` and use `.errors()` which returns structured error info with locations and messages.

7. Q: What's the difference between Pydantic v1 and v2?
   A: v2 has a reworked validator API (`field_validator`, `model_validator`), new parse/dump methods (`model_validate`, `model_dump`), and performance improvements. Some v1 APIs are deprecated — check migration guide when upgrading.

8. Q: Where is Pydantic commonly used in web apps?
   A: Request body validation, response models, configuration parsing, and background job payload validation — FastAPI uses Pydantic heavily.

9. Q: Can Pydantic coerce types? For example, convert numeric strings to ints?
   A: Yes — Pydantic will attempt coercion when possible (e.g., "25" -> 25). Use validators to customize coercion behavior.

10. Q: How would you test Pydantic models?
    A: Write unit tests for valid and invalid inputs, assert `ValidationError` for bad inputs, and validate serialization/deserialization using `model_dump()` / `model_validate()`.


---

If you want, I can:
- run the notebook cells to verify examples (if you want execution),
- add a short FastAPI example showing request validation using a Pydantic model, or
- convert the expanded content into a standalone markdown cheat-sheet.

Which of these would you like next?